# YOLOE -> Any6D Pipeline

Connects YOLOE open-vocabulary detection with Any6D 6D pose estimation.

```
Anchor image (color.png)
        |
   YOLOE text prompt        (master_env venv)
   detects object -> mask
        |
   Any6D register_any6d()   (Docker container)
   registers 6D pose
        |
   4x4 pose matrix
```

**Prerequisites:**
- `source ~/open-vocabulary-6d-pose-yoloe/master_env/bin/activate`
- Docker running with the `any6d` container
- Anchor results downloaded
- Kernel = `master_env`

## Cell 1 — Config & Imports

In [ ]:
import os
import sys
import numpy as np
import cv2
import trimesh
import matplotlib.pyplot as plt
import subprocess

BASE_DIR = os.path.expanduser('~/open-vocabulary-6d-pose-yoloe')
sys.path.insert(0, os.path.join(BASE_DIR, 'utils'))

from any6d_utils import (
    mesh_bbox_corners, project_points,
    draw_3d_bbox, draw_pose_axes,
    mask_overlay, colormap_depth,
    save_fig, axes_legend
)
from pipeline_utils import (
    run_any6d_docker, compute_errors,
    plot_detection, plot_pose_result,
    plot_all_results_summary,
    ANY6D_DIR, YOLOE_MODEL_PATH
)
from yoloe_helpers import yoloe_text_prompt

ANCHOR_DIR = os.path.join(ANY6D_DIR, 'anchor_results', 'dexycb_reference_view_ours')
VIZ_DIR    = os.path.join(BASE_DIR, 'notebooks', 'outputs')
os.makedirs(VIZ_DIR, exist_ok=True)

OBJECTS = [
    {'folder': '006_mustard_bottle',  'prompt': 'mustard bottle',
     'mesh': 'center_mesh_006_mustard_bottle.obj'},
    {'folder': '021_bleach_cleanser', 'prompt': ['white bottle', 'detergent bottle', 'cleaning bottle', 'white plastic bottle', 'laundry bottle'],
     'mesh': 'center_mesh_021_bleach_cleanser.obj'},
    {'folder': '019_pitcher_base',    'prompt': ['blue pitcher', 'blue jug', 'blue container', 'plastic pitcher'],
     'mesh': 'center_mesh_019_pitcher_base.obj'},
    {'folder': '004_sugar_box',       'prompt': 'sugar box',
     'mesh': 'center_mesh_004_sugar_box.obj'},
    {'folder': '005_tomato_soup_can', 'prompt': 'tomato soup can',
     'mesh': 'center_mesh_005_tomato_soup_can.obj'},
    {'folder': '003_cracker_box',     'prompt': 'cracker box',
     'mesh': 'center_mesh_003_cracker_box.obj'},
    {'folder': '010_potted_meat_can', 'prompt': ['yellow box', 'small box', 'square can', 'yellow can', 'small container'],
     'mesh': 'center_mesh_010_potted_meat_can.obj'},
]

print(f'ANCHOR_DIR : {ANCHOR_DIR}')
print(f'VIZ_DIR    : {VIZ_DIR}')
print(f'Objects    : {len(OBJECTS)}')

## Cell 2 — Check Prerequisites

In [ ]:
all_ok = True
print('=== CHECKING PREREQUISITES ===')

print('\n[1] YOLOE model:')
if os.path.exists(YOLOE_MODEL_PATH):
    size_mb = os.path.getsize(YOLOE_MODEL_PATH) / 1e6
    print(f'  OK  yoloe-26l-seg.pt ({size_mb:.0f} MB)')
else:
    print(f'  MISSING: {YOLOE_MODEL_PATH}')
    all_ok = False

print('\n[2] Anchor data (color + depth + mask + mesh + K):')
for obj in OBJECTS:
    folder   = obj['folder']
    obj_path = os.path.join(ANCHOR_DIR, folder)
    required = ['color.png', 'depth.png', 'mask.png', 'K.txt', obj['mesh']]
    missing  = [f for f in required if not os.path.exists(os.path.join(obj_path, f))]
    status   = 'OK' if not missing else f'MISSING: {missing}'
    print(f'  [{"OK" if not missing else "FAIL"}] {folder}  {status}')
    if missing:
        all_ok = False

print('\n[3] Docker + Any6D:')
check_script = """
import sys
sys.path.insert(0, '/workspace/foundationpose/mycpp/build')
import nvdiffrast.torch as dr
from estimater import Any6D
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('DOCKER_OK')
"""
script_path = os.path.join(ANY6D_DIR, '_check.py')
with open(script_path, 'w') as f:
    f.write(check_script)
r = subprocess.run(
    ['docker', 'compose', 'run', '--rm', '--entrypoint', '', 'any6d',
     'bash', '-c', 'cd /workspace && python _check.py'],
    cwd=ANY6D_DIR, capture_output=True, text=True, timeout=60
)
os.remove(script_path)
if 'DOCKER_OK' in r.stdout:
    for line in r.stdout.split('\n'):
        if any(k in line for k in ['GPU:', 'DOCKER_OK']):
            print(f'  {line.strip()}')
else:
    print('  Docker failed')
    all_ok = False

print(f'\n{"="*45}')
print('All prerequisites OK' if all_ok else 'Fix missing items above before running Cell 3')

## Cell 3 — Visualize Input Data

Show colour, depth, and GT mask for all 7 objects before running the pipeline.

In [ ]:
if not os.path.exists(ANCHOR_DIR):
    print('Anchor folder not found')
else:
    fig, axes = plt.subplots(len(OBJECTS), 3, figsize=(15, len(OBJECTS) * 3.5))

    for i, obj in enumerate(OBJECTS):
        obj_path = os.path.join(ANCHOR_DIR, obj['folder'])

        color = cv2.cvtColor(cv2.imread(os.path.join(obj_path, 'color.png')), cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(color)
        axes[i, 0].set_title(f'{obj["folder"]}\nColour', fontsize=8)
        axes[i, 0].axis('off')

        depth = cv2.imread(os.path.join(obj_path, 'depth.png'), cv2.IMREAD_ANYDEPTH).astype(np.float32) / 1000.0
        axes[i, 1].imshow(colormap_depth(depth))
        axes[i, 1].set_title(f'Depth [{depth[depth>0].min():.2f}—{depth.max():.2f}] m', fontsize=8)
        axes[i, 1].axis('off')

        mask      = cv2.imread(os.path.join(obj_path, 'mask.png'), cv2.IMREAD_GRAYSCALE)
        mask_bool = mask > 127
        axes[i, 2].imshow(mask_overlay(color, mask_bool))
        axes[i, 2].set_title(f'GT mask — {mask_bool.sum()} px  prompt: "{obj["prompt"]}"', fontsize=8)
        axes[i, 2].axis('off')

    plt.suptitle('Input data — all 7 objects')
    plt.tight_layout()
    save_fig(fig, os.path.join(VIZ_DIR, 'pipeline_input_data.png'))
    plt.show()

## Cell 4 — Run YOLOE on All 7 Objects

Detect each object using its text prompt and produce a segmentation mask.

In [ ]:
if not all_ok:
    print('Prerequisites not met — fix Cell 2 first')
else:
    from ultralytics import YOLOE
    yoloe_model = YOLOE(YOLOE_MODEL_PATH)
    yoloe_model.to('cuda')

    yoloe_results = {}

    for obj in OBJECTS:
        folder     = obj['folder']
        scene_path = os.path.join(ANCHOR_DIR, folder, 'color.png')

        print(f'\n[YOLOE] {folder}')
        try:
            bbox_xyxy, mask_bool, confidence, _ = yoloe_text_prompt(
                yoloe_model, scene_path, obj['prompt'], conf=0.1
            )
            scene_rgb = cv2.cvtColor(cv2.imread(scene_path), cv2.COLOR_BGR2RGB)
            yoloe_results[folder] = {
                'bbox': bbox_xyxy, 'mask': mask_bool,
                'conf': confidence, 'scene_rgb': scene_rgb, 'success': True
            }
            plot_detection(
                scene_rgb=scene_rgb, mask_bool=mask_bool,
                bbox_xyxy=bbox_xyxy, confidence=confidence,
                text_prompt=obj['prompt'], obj_name=folder,
                viz_dir=VIZ_DIR, save=True
            )
        except Exception as e:
            print(f'  Detection failed: {e}')
            yoloe_results[folder] = {'success': False, 'error': str(e)}

    print('\n=== YOLOE DETECTION SUMMARY ===')
    for folder, res in yoloe_results.items():
        if res['success']:
            print(f'  OK  {folder:<30} conf={res["conf"]:.3f}  mask={res["mask"].sum()} px')
        else:
            print(f'  FAIL  {folder:<30} {res["error"]}')

## Cell 5 — Run Any6D on All 7 Objects

For each object where YOLOE succeeded, run Any6D pose estimation in Docker.

**Expected time: ~5-7 minutes for all 7 objects**

In [ ]:
subprocess.run(['chmod', '-R', '777',
    os.path.join(ANY6D_DIR, 'results', 'pipeline')], capture_output=True)


In [ ]:
if not all_ok:
    print('Prerequisites not met — fix Cell 2 first')
elif not yoloe_results:
    print('No YOLOE results — run Cell 4 first')
else:
    any6d_results = {}

    for obj in OBJECTS:
        folder = obj['folder']

        if not yoloe_results.get(folder, {}).get('success', False):
            print(f'\n[Any6D] Skipping {folder} — YOLOE failed')
            continue

        obj_path   = os.path.join(ANCHOR_DIR, folder)
        mask_bool  = yoloe_results[folder]['mask']
        K          = np.loadtxt(os.path.join(obj_path, 'K.txt'))
        save_path  = os.path.join(ANY6D_DIR, 'results', 'pipeline', folder)

        print(f'\n[Any6D] {folder}  mask={mask_bool.sum()} px')
        try:
            pred_pose = run_any6d_docker(
                color_path=os.path.join(obj_path, 'color.png'),
                depth_path=os.path.join(obj_path, 'depth.png'),
                mask_bool=mask_bool,
                K=K,
                mesh_path=os.path.join(obj_path, obj['mesh']),
                save_path=save_path,
                name=folder, iteration=5
            )
            gt_pose_path = os.path.join(obj_path, f'{folder}_gt_pose.txt')
            gt_pose      = np.loadtxt(gt_pose_path) if os.path.exists(gt_pose_path) else None
            err_t, err_R = compute_errors(pred_pose, gt_pose) if gt_pose is not None else (0.0, 0.0)

            any6d_results[folder] = {
                'pred_pose': pred_pose, 'gt_pose': gt_pose,
                'err_t': err_t, 'err_R': err_R,
                'K': K, 'success': True
            }
            print(f'  T error: {err_t:.2f} cm   R error: {err_R:.2f} deg')

        except Exception as e:
            print(f'  Any6D failed: {e}')
            any6d_results[folder] = {'success': False, 'error': str(e)}

    print('\n=== ANY6D RESULTS SUMMARY ===')
    for folder, res in any6d_results.items():
        if res['success']:
            print(f'  OK  {folder:<30} T={res["err_t"]:.1f}cm  R={res["err_R"]:.1f}deg')
        else:
            print(f'  FAIL  {folder:<30} {res["error"]}')

## Cell 6 — Visualize Pose Results per Object

For each object: predicted pose (blue bbox + axes) vs ground truth (green bbox).

In [ ]:
if not any6d_results:
    print('No Any6D results — run Cell 5 first')
else:
    for obj in OBJECTS:
        folder = obj['folder']
        if not any6d_results.get(folder, {}).get('success', False):
            print(f'Skipping {folder}')
            continue

        res      = any6d_results[folder]
        yolo_res = yoloe_results[folder]
        obj_path = os.path.join(ANCHOR_DIR, folder)
        mesh     = trimesh.load(os.path.join(obj_path, obj['mesh']))

        print(f'\n[VIZ] {folder}')
        plot_pose_result(
            scene_rgb=yolo_res['scene_rgb'], pred_pose=res['pred_pose'],
            gt_pose=res['gt_pose'], K=res['K'],
            mesh=mesh, mask_bool=yolo_res['mask'],
            confidence=yolo_res['conf'],
            err_t=res['err_t'], err_R=res['err_R'],
            obj_name=folder, viz_dir=VIZ_DIR, save=True
        )

## Cell 7 — Summary Plot: All 7 Objects

In [ ]:
if not any6d_results:
    print('No results — run Cell 5 first')
else:
    all_results = []

    for obj in OBJECTS:
        folder = obj['folder']
        if not any6d_results.get(folder, {}).get('success', False):
            continue

        res      = any6d_results[folder]
        yolo_res = yoloe_results[folder]
        obj_path = os.path.join(ANCHOR_DIR, folder)
        mesh     = trimesh.load(os.path.join(obj_path, obj['mesh']))

        corners_3d = mesh_bbox_corners(mesh)
        base       = mask_overlay(yolo_res['scene_rgb'], yolo_res['mask'], alpha=0.25)
        c_pred     = project_points(corners_3d, res['pred_pose'], res['K'])
        img_pose   = draw_pose_axes(
            draw_3d_bbox(base, c_pred, (30, 120, 255)),
            res['pred_pose'], res['K']
        )

        all_results.append({
            'obj_name': folder,
            'err_t':    res['err_t'],
            'err_R':    res['err_R'],
            't_pred':   res['pred_pose'][:3, 3],
            't_gt':     res['gt_pose'][:3, 3] if res['gt_pose'] is not None else None,
            'img_pose': img_pose
        })

    plot_all_results_summary(all_results, VIZ_DIR, save=True)

## Cell 8 — Final Summary

In [ ]:
print('=' * 55)
print('YOLOE -> ANY6D PIPELINE — FINAL SUMMARY')
print('=' * 55)

if not any6d_results:
    print('\n  No results yet — run Cells 4 and 5 first')
else:
    success = [f for f, r in any6d_results.items() if r.get('success')]
    failed  = [f for f, r in any6d_results.items() if not r.get('success')]

    print(f'\n  Completed: {len(success)}/{len(OBJECTS)} objects')

    if success:
        err_ts = [any6d_results[f]['err_t'] for f in success]
        err_Rs = [any6d_results[f]['err_R'] for f in success]
        print(f'\n  {"Object":<30} {"T err (cm)":>12} {"R err (deg)":>12}')
        print(f'  {"-"*56}')
        for f in success:
            res = any6d_results[f]
            print(f'  {f:<30} {res["err_t"]:>12.1f} {res["err_R"]:>12.1f}')
        print(f'  {"-"*56}')
        print(f'  {"MEAN":<30} {np.mean(err_ts):>12.1f} {np.mean(err_Rs):>12.1f}')

    if failed:
        print(f'\n  Failed:')
        for f in failed:
            print(f'  {f}: {any6d_results[f].get("error", "unknown")}')

    print('\n  Output files (notebooks/outputs/):')
    output_files = (
        ['pipeline_input_data.png', 'pipeline_all_results_summary.png'] +
        [f'{f}_yoloe_detection.png' for f in success] +
        [f'{f}_pose_result.png' for f in success]
    )
    for fname in output_files:
        path   = os.path.join(VIZ_DIR, fname)
        status = 'OK' if os.path.exists(path) else 'MISSING'
        print(f'  [{status}] {fname}')